In [1]:
from google.colab import files
uploaded = files.upload()  # select 4-5 Tracked_Video_Actor_XX.zip
print("Uploaded:", list(uploaded.keys()))

Saving Tracked_Video_Actor_01.zip to Tracked_Video_Actor_01.zip
Saving Tracked_Video_Actor_02.zip to Tracked_Video_Actor_02.zip
Saving Tracked_Video_Actor_03.zip to Tracked_Video_Actor_03.zip
Saving Tracked_Video_Actor_04.zip to Tracked_Video_Actor_04.zip
Saving Tracked_Video_Actor_05.zip to Tracked_Video_Actor_05.zip
Uploaded: ['Tracked_Video_Actor_01.zip', 'Tracked_Video_Actor_02.zip', 'Tracked_Video_Actor_03.zip', 'Tracked_Video_Actor_04.zip', 'Tracked_Video_Actor_05.zip']


In [2]:
import os, zipfile

EXTRACT_DIR = "ravdess_video_actors"
os.makedirs(EXTRACT_DIR, exist_ok=True)

for zname in uploaded.keys():
    if zname.lower().endswith(".zip"):
        with zipfile.ZipFile(zname, "r") as z:
            z.extractall(EXTRACT_DIR)

print("Extraction complete:", EXTRACT_DIR)

Extraction complete: ravdess_video_actors


In [3]:
import os
import pandas as pd

EMOTION_MAP = {
    1: "neutral", 2: "calm", 3: "happy", 4: "sad",
    5: "angry", 6: "fearful", 7: "disgust", 8: "surprised"
}
STRESS_EMOTIONS = {"angry", "fearful", "sad"}
NOSTRESS_EMOTIONS = {"neutral", "calm", "happy"}

def parse_ravdess_filename(fname: str):
    base = os.path.splitext(os.path.basename(fname))[0]
    parts = base.split("-")
    if len(parts) != 7:
        return None
    try:
        emotion_code = int(parts[2])
        actor_id = int(parts[6])
    except:
        return None
    emotion = EMOTION_MAP.get(emotion_code)
    if emotion is None:
        return None
    return emotion, actor_id

video_paths = []
for root, _, files_ in os.walk(EXTRACT_DIR):
    for f in files_:
        if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv")):
            video_paths.append(os.path.join(root, f))

rows = []
for p in video_paths:
    parsed = parse_ravdess_filename(os.path.basename(p))
    if not parsed:
        continue
    emotion, actor = parsed

    if emotion in STRESS_EMOTIONS:
        label = 1
    elif emotion in NOSTRESS_EMOTIONS:
        label = 0
    else:
        continue  # drop disgust/surprised

    rows.append({"path": p, "emotion": emotion, "label": label, "actor": actor})

df = pd.DataFrame(rows)
print("Usable videos:", len(df))
print("Actors:", sorted(df["actor"].unique().tolist()))
print("Label counts:\n", df["label"].value_counts())
print("Emotion counts:\n", df["emotion"].value_counts())
df.head()

Usable videos: 440
Actors: [1, 2, 3, 4, 5]
Label counts:
 label
1    240
0    200
Name: count, dtype: int64
Emotion counts:
 emotion
calm       80
happy      80
sad        80
angry      80
fearful    80
neutral    40
Name: count, dtype: int64


,path,emotion,label,actor
0,ravdess_video_actors/Tracked_Video_Actor_02/01...,calm,0,2
1,ravdess_video_actors/Tracked_Video_Actor_02/01...,happy,0,2
2,ravdess_video_actors/Tracked_Video_Actor_02/01...,neutral,0,2
3,ravdess_video_actors/Tracked_Video_Actor_02/01...,sad,1,2
4,ravdess_video_actors/Tracked_Video_Actor_02/01...,calm,0,2


In [4]:
actors = sorted(df["actor"].unique().tolist())
print("Available actors:", actors)

test_actor = actors[-1]   # change this if you want
df_train = df[df["actor"] != test_actor].reset_index(drop=True)
df_test  = df[df["actor"] == test_actor].reset_index(drop=True)

print("Test actor:", test_actor)
print("Train size:", len(df_train), "Test size:", len(df_test))
print("Train labels:", df_train["label"].value_counts().to_dict())
print("Test labels:", df_test["label"].value_counts().to_dict())

Available actors: [1, 2, 3, 4, 5]
Test actor: 5
Train size: 352 Test size: 88
Train labels: {1: 192, 0: 160}
Test labels: {1: 48, 0: 40}


In [5]:
!pip -q install opencv-python tqdm

In [6]:
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMG_SIZE = 160
NUM_FRAMES = 12

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.2, contrast=0.2)], p=0.4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def sample_frames(video_path, num_frames=NUM_FRAMES):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if length <= 0:
        cap.release()
        return None
    idxs = np.linspace(0, max(0, length - 1), num_frames).astype(int)
    frames = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    cap.release()
    return frames if len(frames) else None

class RavdessVideoDataset(Dataset):
    def __init__(self, df, is_train=True):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        frames = sample_frames(row["path"], NUM_FRAMES)

        tf = train_tf if self.is_train else test_tf

        if frames is None:
            x = torch.zeros((NUM_FRAMES, 3, IMG_SIZE, IMG_SIZE), dtype=torch.float32)
        else:
            imgs = [tf(Image.fromarray(f)) for f in frames]
            while len(imgs) < NUM_FRAMES:
                imgs.append(imgs[-1])
            x = torch.stack(imgs[:NUM_FRAMES], dim=0)

        y = torch.tensor(int(row["label"]), dtype=torch.long)
        return x, y

In [7]:
import torch.nn as nn
from torchvision.models import resnet18

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

class FrameAvgResNet(nn.Module):
    def __init__(self, unfreeze_layer3=False):
        super().__init__()
        self.backbone = resnet18(weights="DEFAULT")
        self.backbone.fc = nn.Identity()

        # Freeze all
        for p in self.backbone.parameters():
            p.requires_grad = False

        # Unfreeze layer4 (recommended)
        for p in self.backbone.layer4.parameters():
            p.requires_grad = True

        # Optional: unfreeze layer3 for extra learning (slower, sometimes better)
        if unfreeze_layer3:
            for p in self.backbone.layer3.parameters():
                p.requires_grad = True

        self.classifier = nn.Linear(512, 2)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B*T, C, H, W)
        feats = self.backbone(x)
        logits = self.classifier(feats)
        logits = logits.view(B, T, 2).mean(dim=1)
        return logits

model = FrameAvgResNet(unfreeze_layer3=False).to(device)

Device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 143MB/s]


In [8]:
import numpy as np
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report

train_loader = DataLoader(RavdessVideoDataset(df_train, is_train=True), batch_size=4, shuffle=True, num_workers=2)
test_loader  = DataLoader(RavdessVideoDataset(df_test,  is_train=False), batch_size=4, shuffle=False, num_workers=2)

# Class weights (helps if biased)
counts = np.bincount(df_train["label"].values)
w0 = counts.sum() / (2 * counts[0])
w1 = counts.sum() / (2 * counts[1])
class_w = torch.tensor([w0, w1], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_w)
print("Train label counts:", counts, "weights:", class_w.detach().cpu().numpy())

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

def eval_model():
    model.eval()
    ys, ps = [], []
    total_loss = 0.0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += float(loss.item()) * x.size(0)
            pred = torch.argmax(logits, dim=1).cpu().numpy()
            ys.extend(y.cpu().numpy())
            ps.extend(pred)
    ys, ps = np.array(ys), np.array(ps)
    return total_loss / len(test_loader.dataset), ys, ps

best_acc = -1.0
patience = 2
bad_epochs = 0

EPOCHS = 8
for ep in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for x, y in tqdm(train_loader, leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item()) * x.size(0)

    tr_loss = total_loss / len(train_loader.dataset)
    te_loss, y_true, y_pred = eval_model()

    acc = (y_true == y_pred).mean()
    print(f"\nEpoch {ep}: train_loss={tr_loss:.4f} test_loss={te_loss:.4f} acc={acc:.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4))

    # Early stopping
    if acc > best_acc + 1e-4:
        best_acc = acc
        bad_epochs = 0
        torch.save(model.state_dict(), "best_video_model.pt")
        print("Saved best model.")
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print("Early stopping.")
            break

print("Best test accuracy:", best_acc)

Train label counts: [160 192] weights: [1.1       0.9166667]



Epoch 1: train_loss=0.4212 test_loss=0.4357 acc=0.8068
Confusion Matrix:
 [[40  0]
 [17 31]]
              precision    recall  f1-score   support

           0     0.7018    1.0000    0.8247        40
           1     1.0000    0.6458    0.7848        48

    accuracy                         0.8068        88
   macro avg     0.8509    0.8229    0.8048        88
weighted avg     0.8644    0.8068    0.8030        88

Saved best model.



Epoch 2: train_loss=0.2555 test_loss=1.4380 acc=0.5114
Confusion Matrix:
 [[40  0]
 [43  5]]
              precision    recall  f1-score   support

           0     0.4819    1.0000    0.6504        40
           1     1.0000    0.1042    0.1887        48

    accuracy                         0.5114        88
   macro avg     0.7410    0.5521    0.4195        88
weighted avg     0.7645    0.5114    0.3986        88




Epoch 3: train_loss=0.2360 test_loss=1.2141 acc=0.6818
Confusion Matrix:
 [[40  0]
 [28 20]]
              precision    recall  f1-score   support

           0     0.5882    1.0000    0.7407        40
           1     1.0000    0.4167    0.5882        48

    accuracy                         0.6818        88
   macro avg     0.7941    0.7083    0.6645        88
weighted avg     0.8128    0.6818    0.6576        88

Early stopping.
Best test accuracy: 0.8068181818181818
